In [2]:
import os
import importlib
#os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
import time

device1 = 'cuda:0'
device2 = 'cuda:1'

data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
data_dir = '../data'

In [3]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(11645, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,b11f0628-a295-410b-b63e-9a6aae0fc415,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,0209e925-32e1-4607-8b4f-8c0795b93383,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,a2d355c4-9e28-4325-83d5-45013a6d415d,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,3343fc27-96f0-41bf-953a-4b42bfb07bb1,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,7d7c9b56-f103-4f07-9288-aeb9f3e8560c,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [4]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:09<00:00,  2.28s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [5]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.03it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [6]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)

    # query_embedding = query_embedding.unsqueeze(0)
    #print(query_embedding)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    #print(top_results)
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return res

In [10]:
from evaluation import evaluate
from tqdm import tqdm
import nltk

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    
    # print("Retrieved doc :")
    # for j in range(len(results)):
    #     print(f"\tRank {j} : {results[j]}")
        
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(retrieved_docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
    print(candidate)
    scores=evaluate(candidate, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores_df.mean())
    
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  5%|▌         | 1/20 [00:04<01:30,  4.77s/it]

['Cristiano Ronaldo has the highest goals in world football, with a total of 133 international goals.']
rougeLsum    32.142857
length       16.000000
str_em        0.000000
ovscore       0.000000
dtype: float64


 10%|█         | 2/20 [00:09<01:21,  4.50s/it]

['Simon & Garfunkel.']
rougeLsum    19.461259
length        9.500000
str_em       33.333333
ovscore      10.629880
dtype: float64


 15%|█▌        | 3/20 [00:14<01:27,  5.12s/it]

...

 95%|█████████▌| 19/20 [01:47<00:06,  6.39s/it]

['Heath Ledger plays Patrick Verona in the 1999 film "10 Things I Hate About You".']
rougeLsum    29.747783
length       24.105263
str_em       32.894737
ovscore      25.256214
dtype: float64


100%|██████████| 20/20 [01:52<00:00,  5.61s/it]

['Yes, Microsoft Live Movie Maker is an example of free software, as it was included with Windows operating systems and later available as part of the Windows Live Essentials suite, which was free to download.']
rougeLsum    29.831822
length       24.650000
str_em       31.250000
ovscore      23.993403
dtype: float64


rougeLsum    29.831822
length       24.650000
str_em       31.250000
ovscore      23.993403
dtype: float64

In [20]:
### old scores
# rougeLsum    28.315600
# length       64.359110
# str_em       19.200212
# ovscore      14.931574
# dtype: float64

rougeLsum    28.315600
length       64.359110
str_em       19.200212
ovscore      14.931574
dtype: float64

In [8]:
scores_df.to_csv(f'./results/baseline_results_{time.strftime("%Y%m%d-%H%M%S")}.csv', index=False)

: 